In [21]:
import requests
from bs4 import BeautifulSoup
import time
import re

class TranscriptScraper:
    def __init__(self):
        self.session = requests.Session()
        # Set a user agent to appear as a regular browser
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
    
    def scrape_transcript(self, url, delay=1):
        """
        Scrape transcript text from SubsLikeScript URL
        
        Args:
            url (str): The SubsLikeScript URL
            delay (int): Delay between requests in seconds
            
        Returns:
            dict: Dictionary containing transcript data
        """
        try:
            # Add delay to be respectful to the server
            time.sleep(delay)
            
            # Make request
            response = self.session.get(url)
            response.raise_for_status()
            
            # Parse HTML
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract title
            title_element = soup.find('h1')
            title = title_element.text.strip() if title_element else "Unknown Title"
            
            # Find the transcript container
            # SubsLikeScript typically uses a div with class containing "transcript" or similar
            transcript_div = soup.find('div', class_=re.compile(r'transcript|full-script|script-content', re.I))
            
            if not transcript_div:
                # Try alternative selectors
                transcript_div = soup.find('article') or soup.find('div', {'id': 'content'})
            
            if not transcript_div:
                raise Exception("Could not find transcript content on the page")
            
            # Extract text content
            transcript_text = transcript_div.get_text(separator='\n', strip=True)
            
            # Clean up the text
            transcript_text = self._clean_transcript(transcript_text)
            
            # Split into sentences for emotion analysis
            sentences = self._extract_sentences(transcript_text)
            
            return {
                'title': title,
                'url': url,
                'full_transcript': transcript_text,
                'sentences': sentences,
                'sentence_count': len(sentences)
            }
            
        except requests.RequestException as e:
            print(f"Request error: {e}")
            return None
        except Exception as e:
            print(f"Parsing error: {e}")
            return None
    
    def _clean_transcript(self, text):
        """Clean up transcript text"""
        # Remove extra whitespace
        text = re.sub(r'\n\s*\n', '\n\n', text)
        text = re.sub(r' +', ' ', text)
        
        # Remove common non-dialogue elements
        lines_to_remove = [
            r'^\s*\[.*?\]\s*$',  # Stage directions in brackets
            r'^\s*\(.*?\)\s*$',  # Parenthetical notes
            r'^\s*♪.*♪\s*$',     # Music notes
            r'^\s*www\..*$',     # Website URLs
            r'^\s*SubsLikeScript.*$',  # Website attribution
        ]
        
        lines = text.split('\n')
        cleaned_lines = []
        
        for line in lines:
            should_keep = True
            for pattern in lines_to_remove:
                if re.match(pattern, line, re.IGNORECASE):
                    should_keep = False
                    break
            if should_keep and line.strip():
                cleaned_lines.append(line.strip())
        
        return '\n'.join(cleaned_lines)
    
    def _extract_sentences(self, text):
        """Extract sentences for emotion analysis, combining incomplete sentences"""
        lines = text.split('\n')
        sentences = []
        current_sentence = ""
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # Remove speaker names (everything before the first colon)
            if ':' in line:
                dialogue_text = line.split(':', 1)[1].strip()
            else:
                dialogue_text = line.strip()
            
            # Skip if it's too short or looks like stage direction
            if len(dialogue_text) < 3:
                continue
                
            # Add to current sentence
            if current_sentence:
                current_sentence += " " + dialogue_text
            else:
                current_sentence = dialogue_text
            
            # Check if sentence is complete
            if self._is_complete_sentence(current_sentence):
                sentences.append(current_sentence.strip())
                current_sentence = ""
        
        # Add any remaining sentence
        if current_sentence.strip():
            sentences.append(current_sentence.strip())
        
        return sentences
    
    def _is_complete_sentence(self, text):
        """Check if a sentence ends with proper punctuation (not comma)"""
        if not text:
            return False
        
        # Remove trailing whitespace and quotes
        text = text.rstrip().rstrip('"\'')
        
        # Check if it ends with sentence-ending punctuation
        sentence_endings = ['.', '!', '?', '...', '."', '!"', '?"', ".'", "!'", "?'"]
        
        # Also check for other endings that indicate completeness
        other_endings = [':', ';', '—', '--']
        
        return any(text.endswith(ending) for ending in sentence_endings + other_endings)
    
    def save_transcript(self, transcript_data, filename):
        """Save transcript data to file"""
        if not transcript_data:
            print("No transcript data to save")
            return
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"Title: {transcript_data['title']}\n")
            f.write(f"URL: {transcript_data['url']}\n")
            f.write(f"Total sentences: {transcript_data['sentence_count']}\n")
            f.write("=" * 50 + "\n\n")
            f.write(transcript_data['full_transcript'])
        
        print(f"Transcript saved to {filename}")
        
        # Also save clean sentences for emotion analysis
        sentences_filename = filename.replace('.txt', '_sentences.txt')
        with open(sentences_filename, 'w', encoding='utf-8') as f:
            for i, sentence in enumerate(transcript_data['sentences'], 1):
                f.write(f"{i}. {sentence}\n")
        
        print(f"Clean sentences saved to {sentences_filename}")

In [24]:
scraper = TranscriptScraper()
    
url = "https://subslikescript.com/series/Kitchen_Nightmares-983514/season-1/episode-3-The_Mixing_Bowl"

print("Scraping transcript...")
transcript_data = scraper.scrape_transcript(url)

if transcript_data:
    print(f"Successfully scraped: {transcript_data['title']}")
    print(f"Found {transcript_data['sentence_count']} sentences")
    
    # Save to file
    scraper.save_transcript(transcript_data, "C:\\Users\\filip\\Desktop\\kitchen_nightmares_transcript.txt")
    
    # Preview first few sentences
    print("\nFirst 5 sentences:")
    for i, sentence in enumerate(transcript_data['sentences'][:5], 1):
        print(f"{i}. {sentence}")
        
    # Show an example of combined sentences
    print("\nExample of longer sentences:")
    for i, sentence in enumerate(transcript_data['sentences'][:10], 1):
        if len(sentence.split()) > 15:  # Show longer combined sentences
            print(f"{i}. {sentence}")
            break
else:
        print("Failed to scrape transcript")

Scraping transcript...
Successfully scraped: Kitchen Nightmares (2007–2014): Season 1, Episode 3  - The Mixing Bowl - full transcript
Found 1136 sentences
Transcript saved to C:\Users\filip\Desktop\kitchen_nightmares_transcript.txt
Clean sentences saved to C:\Users\filip\Desktop\kitchen_nightmares_transcript_sentences.txt

First 5 sentences:
1. For the last three years Chef Ramsay has whipped aspiring chefs into shape on Hell's Kitchen.
2. this is painful!
3. Get out!
4. Out!
5. Now Gordon Ramsay, the most successful restaurateur on the planet with critically-acclaimed restaurants in London, Dubai, Tokyo, and New York, is criss-crossing America for the most difficult assignment of his career.

Example of longer sentences:
1. For the last three years Chef Ramsay has whipped aspiring chefs into shape on Hell's Kitchen.
